**Install PANNA package and requirements:**


In [2]:
import os, sys, subprocess
from pathlib import Path
from IPython.display import Code
from ase.io import read
from ase.visualize import view
from scripts import *

main_dir = find_project_root()
print(main_dir)

/home/mina-joojoo/Desktop/temporary/CECAM_LATTE


In [ ]:
!{sys.executable} -m pip install {main_dir}/PANNA

**Let's visualize the Carbon dataset:**

In [ ]:
file = os.path.join(main_dir, 'dia_100/datasets/training_100_dia.xyz')
trajectory = read(file, index=":")
view(trajectory)

**Step 1: train LATTE on a small diamond dataset.**

training set: 80 atomic structures, test set: 20 structures.

In [ ]:
os.chdir(f'{main_dir}/dia_100/training')
Code(filename='train.ini')

In [ ]:
# step 1: train the model
!{sys.executable} -m panna.train_jax -c train.ini

In [ ]:
# step 2: Visualize the training process
fig, axs = plt.subplots(1, 2, figsize = (12, 5))
plot_training(f'{main_dir}/dia_100/training', axs)
plot_setup(fig, axs, log=True, xlim=2000, 
            ylim=0.1, fylim=0.3,
            title='Trained on 100 Diamond')

In [ ]:
# step 3: let's make prediction on Diamond test set. Is this model able to predict more diverse dataset?
os.chdir(f'{main_dir}/dia_100/test_dia')
!{sys.executable} -m panna.train_jax -c train.ini
os.chdir(f'{main_dir}/dia_100/test_div')
!{sys.executable} -m panna.train_jax -c train.ini

In [ ]:
# step 4: measure the prediction error
path = f'{main_dir}/dia_100/test_dia'
epoch = 50
Force_mae, Energy_mae= Force_MAE(path, epoch), Energy_MAE(path, epoch) 
print(f'At epoch {epoch}: \n Force_MAE: {Force_mae:.3f} \n Energy_MAE: {Energy_mae:.3f}')
path = f'{main_dir}/dia_100/test_div'
epoch = 50
Force_mae, Energy_mae= Force_MAE(path, epoch), Energy_MAE(path, epoch) 
print(f'At epoch {epoch}: \n Force_MAE: {Force_mae:.3f} \n Energy_MAE: {Energy_mae:.3f}')

In [ ]:
# visualize all the trainings
dirs = ['dia_100', 'dia_500','dia_1k','dia_5k','div_100','div_500','div_1k','div_5k']
colors = ['r', 'b', 'g', 'k', 'c', 'm', 'orange', 'lime']
fig, axs = plt.subplots(1, 2, figsize = (12, 5))
for d, c in zip(dirs, colors):
    plot_training(os.path.join(str(main_dir),d ,'training'), axs, train = False, label = d, color = c)
plot_setup(fig, axs, log=False, fylim = 0.6, xlim=1100, title='All trainings')

Data efficiency:

In [ ]:
x = [100, 500, 1000, 5000] # size of dataset
p = main_dir
dia_dirs = ['dia_100', 'dia_500','dia_1k', 'dia_5k']
dia_epochs = [150, 250, 250, 1000]
dia_dia_errors = []
dia_div_errors = []

for d, e in zip(dia_dirs, dia_epochs):
    path_aa = os.path.join(p, d, 'test_dia')
    aa = Force_MAE(path_aa, f'epoch_{e}_step_{e*1000}_forces.dat')
    dia_dia_errors.append(aa)
    path_av = os.path.join(p, d, 'test_div')
    av = Force_MAE(path_av, f'epoch_{e}_step_{e*1000}_forces.dat')
    dia_div_errors.append(av)

div_dirs = ['div_100', 'div_500','div_1k', 'div_5k']
div_epochs = [50, 50, 50, 1000]
div_div_errors = []
div_dia_errors = []


for dv, ev in zip(div_dirs, div_epochs):
    path_vv = os.path.join(p, dv, 'test_div')
    vv = Force_MAE(path_vv, f'epoch_{ev}_step_{ev*1000}_forces.dat')
    div_div_errors.append(aa)
    path_va = os.path.join(p, dv, 'test_dia')
    va = Force_MAE(path_va, f'epoch_{ev}_step_{ev*1000}_forces.dat')
    div_dia_errors.append(av)